In [1]:
import torch
import sys

print(f"Python 版本: {sys.version.split()[0]}")
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 版本: {torch.version.cuda}")

Python 版本: 3.12.12
PyTorch 版本: 2.10.0+cu128
CUDA 版本: 12.8


In [2]:
import torch
import os

# 1. 自動偵測 Colab 當前的環境版本
cuda_version = "".join(torch.version.cuda.split("."))
torch_version = torch.__version__.split("+")[0]
python_version = "cp310" # Colab 預設是 python 3.10

print(f"🔍 偵測到環境: PyTorch {torch_version}, CUDA {cuda_version}")

# 2. 自動組合官方 Release 的「懶人包 (Wheel)」下載網址
# (這裡使用穩定的 1.2.0 版本，相容性最高)
causal_conv_url = f"https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.2.0/causal_conv1d-1.2.0+cu{cuda_version}torch{torch_version}cxx11abiFalse-{python_version}-{python_version}-linux_x86_64.whl"
mamba_url = f"https://github.com/state-spaces/mamba/releases/download/v1.2.0/mamba_ssm-1.2.0+cu{cuda_version}torch{torch_version}cxx11abiFalse-{python_version}-{python_version}-linux_x86_64.whl"

# 3. 執行光速安裝
print("🚀 開始下載並安裝懶人包 (大約只需 5~10 秒)...")
os.system(f"pip install {causal_conv_url}")
os.system(f"pip install {mamba_url}")

print("✅ 光速安裝指令已送出！請執行下一格測試是否成功引入。")

🔍 偵測到環境: PyTorch 2.10.0, CUDA 128
🚀 開始下載並安裝懶人包 (大約只需 5~10 秒)...
✅ 光速安裝指令已送出！請執行下一格測試是否成功引入。


In [3]:
try:
    from mamba_ssm import Mamba
    print("🎉 太神啦！Mamba 套件已成功載入，運算單元保住了！")
except ImportError as e:
    print(f"❌ 載入失敗: {e}")

❌ 載入失敗: No module named 'mamba_ssm'


In [4]:
import os

print("🔧 啟動 Mamba 編譯優化設定...")

# 1. 召喚加速神器 Ninja
os.system("pip install ninja")

# 2. 限定 GPU 架構 (從你剛剛的報錯日誌得知是 sm_87，對應 8.7)
# 這一行是大幅縮短編譯時間的魔法，它告訴編譯器：「別管其他顯卡了，專心針對 8.7 編譯就好！」
os.environ["TORCH_CUDA_ARCH_LIST"] = "8.7"

# 3. 限制編譯的工人數量
# 避免 Colab 的記憶體被瞬間抽乾而崩潰，我們限制最多 4 個執行緒同時編譯
os.environ["MAX_JOBS"] = "4"

# 4. 指定編譯產生器
os.environ["CMAKE_GENERATOR"] = "Ninja"

print("✅ 環境變數設定完畢！準備進入硬核編譯階段。")

🔧 啟動 Mamba 編譯優化設定...
✅ 環境變數設定完畢！準備進入硬核編譯階段。


In [5]:
# 加上 --no-build-isolation 是為了確保它使用我們目前系統上正確的 PyTorch 版本
!pip install -v causal-conv1d>=1.2.0 --no-build-isolation
!pip install -v mamba-ssm --no-build-isolation

串流輸出內容已截斷至最後 5000 行。
  ptxas info    : Compile time = 27.269 ms
  ptxas info    : Compiling entry function '_Z25selective_scan_fwd_kernelI32Selective_Scan_fwd_kernel_traitsILi32ELi8ELi1ELb1ELb1ELb0ELb1EffEEv13SSMParamsBase' for 'sm_72'
  ptxas info    : Function properties for _Z25selective_scan_fwd_kernelI32Selective_Scan_fwd_kernel_traitsILi32ELi8ELi1ELb1ELb1ELb0ELb1EffEEv13SSMParamsBase
      0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
  ptxas info    : Used 72 registers, used 1 barriers, 552 bytes cmem[0]
  ptxas info    : Compile time = 30.063 ms
  ptxas info    : Compiling entry function '_Z25selective_scan_fwd_kernelI32Selective_Scan_fwd_kernel_traitsILi32ELi8ELi1ELb1ELb1ELb1ELb0EffEEv13SSMParamsBase' for 'sm_72'
  ptxas info    : Function properties for _Z25selective_scan_fwd_kernelI32Selective_Scan_fwd_kernel_traitsILi32ELi8ELi1ELb1ELb1ELb1ELb0EffEEv13SSMParamsBase
      0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
  ptxas info    : Used 83

In [9]:
import os
from google.colab import drive

print("🚀 啟動 Mamba 專武收割程式...")

# 1. 確保雲端硬碟已經掛載
drive.mount('/content/drive')

# 2. 在 Drive 建立一個 T4 專屬的軍火庫資料夾
# 加上 T4 標籤，以後如果你又煉了 L4 的版本才不會搞混
wheel_dir = '/content/drive/MyDrive/Mamba_Wheels_T4'
os.makedirs(wheel_dir, exist_ok=True)

# 3. 去 Colab 的底層快取資料夾，把編譯好的 causal-conv1d 和 mamba-ssm 挖出來複製過去
print(f"🔍 正在將 .whl 檔案備份至: {wheel_dir}")
os.system(f"find ~/.cache/pip/wheels/ -name '*causal_conv1d*.whl' -exec cp {{}} {wheel_dir} \\;")
os.system(f"find ~/.cache/pip/wheels/ -name '*mamba_ssm*.whl' -exec cp {{}} {wheel_dir} \\;")

# 4. 驗證戰利品
print("\n✅ 備份完成！請確認資料夾內是否有這兩個檔案：")
os.system(f"ls -lh {wheel_dir}")

🚀 啟動 Mamba 專武收割程式...
Mounted at /content/drive
🔍 正在將 .whl 檔案備份至: /content/drive/MyDrive/Mamba_Wheels_T4

✅ 備份完成！請確認資料夾內是否有這兩個檔案：


0

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
from tqdm.auto import tqdm

# 🏆 1. 載入我們千辛萬苦裝好的真・Mamba
from mamba_ssm import Mamba

# ==========================================
# 🧠 2. 核心模型架構
# ==========================================
class DenoiseNetwork(nn.Module):
    def __init__(self, pred_length=1, context_dim=128, hidden_dim=256):
        super().__init__()
        self.time_embed = nn.Sequential(
            nn.Linear(1, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, hidden_dim)
        )
        self.net = nn.Sequential(
            nn.Linear(pred_length + hidden_dim + context_dim, hidden_dim * 2),
            nn.SiLU(),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, pred_length)
        )

    def forward(self, x_t, t, context):
        t_emb = self.time_embed(t)
        nn_input = torch.cat([x_t, t_emb, context], dim=-1)
        return self.net(nn_input)

class MicroMamba(nn.Module):
    def __init__(self, input_dim=2, d_model=64, context_dim=128):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        # 這裡直接啟用真正的 Mamba！
        self.mamba = Mamba(d_model=d_model, d_state=16, d_conv=4, expand=2)
        self.output_proj = nn.Linear(d_model, context_dim)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.mamba(x)
        last_step_features = x[:, -1, :]
        return self.output_proj(last_step_features)

class MarketMambaDiffusion(nn.Module):
    def __init__(self, seq_length=270, pred_length=1, input_dim=2, context_dim=128):
        super().__init__()
        self.conditioner = MicroMamba(input_dim, d_model=64, context_dim=context_dim)
        self.denoiser = DenoiseNetwork(pred_length, context_dim, hidden_dim=256)

    def compute_loss(self, x_history, y_future):
        batch_size = x_history.shape[0]
        context = self.conditioner(x_history)
        t = torch.rand(batch_size, 1, device=x_history.device)
        noise = torch.randn_like(y_future)
        x_t = torch.sqrt(1 - t) * y_future + torch.sqrt(t) * noise
        predicted_noise = self.denoiser(x_t, t, context)
        return F.mse_loss(predicted_noise, noise)

# ==========================================
# 📊 3. 資料管線 (讀取台積電真實資料)
# ==========================================
class RealMarketDataset(Dataset):
    def __init__(self, df, seq_length=270, pred_length=1):
        features = df[['Log_Ret', 'Log_Vol']].values
        self.features = torch.tensor(features, dtype=torch.float32)
        self.seq_length = seq_length
        self.pred_length = pred_length

    def __len__(self):
        return len(self.features) - self.seq_length - self.pred_length + 1

    def __getitem__(self, idx):
        x = self.features[idx : idx + self.seq_length]
        y = self.features[idx + self.seq_length : idx + self.seq_length + self.pred_length, 0]
        return x, y

# 請確認你的 Google Drive 路徑正確
file_path = '/content/drive/MyDrive/MarketMamba_Database/Micro_5min/2330_5m.parquet'
print("讀取資料中...")
df_real = pd.read_parquet(file_path)
dataset = RealMarketDataset(df_real, seq_length=270)
# batch_size=256 讓 T4 顯卡可以跑得更有效率
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

# ==========================================
# 🚀 4. 終極訓練迴圈 (Training Loop)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 啟動訓練！目前使用的裝置: {device}")

model = MarketMambaDiffusion().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

EPOCHS = 5
model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for x_batch, y_batch in progress_bar:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        loss = model.compute_loss(x_batch, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'Loss': f"{loss.item():.6f}"})

    print(f"✅ Epoch {epoch+1} 完成! 平均 Loss: {total_loss/len(dataloader):.6f}")

print("🎉 恭喜！Mamba 引擎與 Diffusion 完美融合，真實資料測試成功！")

讀取資料中...
🔥 啟動訓練！目前使用的裝置: cuda


Epoch 1/5:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Epoch 1 完成! 平均 Loss: 1.017033


Epoch 2/5:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Epoch 2 完成! 平均 Loss: 0.931984


...

✅ Epoch 3 完成! 平均 Loss: 0.880056


Epoch 4/5:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Epoch 4 完成! 平均 Loss: 0.836199


Epoch 5/5:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Epoch 5 完成! 平均 Loss: 0.704893
🎉 恭喜！Mamba 引擎與 Diffusion 完美融合，真實資料測試成功！


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import os
from tqdm.auto import tqdm

# ==========================================
# 📊 1. 全市場防跨界資料集 (MultiStock Dataset)
# ==========================================
class MultiStockMicroDataset(Dataset):
    def __init__(self, data_dir, seq_length=270, pred_length=1):
        self.seq_length = seq_length
        self.pred_length = pred_length
        self.x_data = []
        self.y_data = []

        file_list = [f for f in os.listdir(data_dir) if f.endswith('.parquet')]
        print(f"🔍 找到 {len(file_list)} 檔股票資料，開始載入記憶體 (這可能需要幾分鐘)...")

        for file_name in tqdm(file_list, desc="讀取全市場 Parquet"):
            file_path = os.path.join(data_dir, file_name)
            try:
                df = pd.read_parquet(file_path)
                features = df[['Log_Ret', 'Log_Vol']].values
                total_len = len(features)

                if total_len < (seq_length + pred_length):
                    continue

                valid_windows = total_len - seq_length - pred_length + 1
                for i in range(valid_windows):
                    self.x_data.append(features[i : i + seq_length])
                    # 目標是預測未來的 Log_Ret (index 0)
                    self.y_data.append(features[i + seq_length : i + seq_length + pred_length, 0])
            except Exception:
                pass # 忽略損壞的檔案

        print(f"⏳ 正在轉換為 Pytorch Tensor (準備榨乾 GPU)...")
        self.x_data = torch.tensor(np.array(self.x_data), dtype=torch.float32)
        self.y_data = torch.tensor(np.array(self.y_data), dtype=torch.float32)
        print(f"✅ 載入完成！全市場共切出 {len(self.x_data)} 組滑動窗口。")

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        return self.x_data[idx], self.y_data[idx]

# 設定你的資料夾路徑
data_dir = '/content/drive/MyDrive/MarketMamba_Database/Micro_5min'
# 設定模型存檔路徑
save_path = '/content/drive/MyDrive/MarketMamba_Database/MarketMamba_V1.pth'

# 啟動 DataLoader！(因為資料變多了，Batch Size 我們維持 256 或開到 512 也可以)
global_dataset = MultiStockMicroDataset(data_dir, seq_length=270)
global_dataloader = DataLoader(global_dataset, batch_size=256, shuffle=True)

# ==========================================
# 🚀 2. 啟動全市場無人值守訓練
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🔥 雙引擎點火！準備訓練全市場資料。目前裝置: {device}")

# (這裡假設你上一格已經載入定義好的 MarketMambaDiffusion 類別了)
model = MarketMambaDiffusion().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

EPOCHS = 10 # 全市場資料很大，我們先讓他跑 10 個 Epoch 試水溫
model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0
    progress_bar = tqdm(global_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for x_batch, y_batch in progress_bar:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        loss = model.compute_loss(x_batch, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'Loss': f"{loss.item():.6f}"})

    avg_loss = total_loss/len(global_dataloader)
    print(f"✅ Epoch {epoch+1} 完成! 均方誤差 (Loss): {avg_loss:.6f}")

    # 💾 核心動作：每跑完一個 Epoch 就存檔！
    torch.save(model.state_dict(), save_path)
    print(f"💾 模型已自動備份至 Drive: {save_path}")

print("🎉 全市場微觀訓練任務圓滿結束！")

🔍 找到 1747 檔股票資料，開始載入記憶體 (這可能需要幾分鐘)...


讀取全市場 Parquet:   0%|          | 0/1747 [00:00<?, ?it/s]

⏳ 正在轉換為 Pytorch Tensor (準備榨乾 GPU)...
✅ 載入完成！全市場共切出 3246892 組滑動窗口。

🔥 雙引擎點火！準備訓練全市場資料。目前裝置: cuda


Epoch 1/10:   0%|          | 0/12684 [00:00<?, ?it/s]

✅ Epoch 1 完成! 均方誤差 (Loss): 0.021459
💾 模型已自動備份至 Drive: /content/drive/MyDrive/MarketMamba_Database/MarketMamba_V1.pth


...

✅ Epoch 3 完成! 均方誤差 (Loss): 0.000661
💾 模型已自動備份至 Drive: /content/drive/MyDrive/MarketMamba_Database/MarketMamba_V1.pth


Epoch 4/10:   0%|          | 0/12684 [00:00<?, ?it/s]

✅ Epoch 4 完成! 均方誤差 (Loss): 0.000580
💾 模型已自動備份至 Drive: /content/drive/MyDrive/MarketMamba_Database/MarketMamba_V1.pth


Epoch 5/10:   0%|          | 0/12684 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ==========================================
# 1. 條件去噪網路 (Denoise Network)
# ==========================================
# 這是 Diffusion 的苦力，負責把「雜訊」還原成「真實的未來股價」
class DenoiseNetwork(nn.Module):
    def __init__(self, pred_length=1, context_dim=128, hidden_dim=256):
        super().__init__()
        # 輸入包含三部分：被加了雜訊的股價 (x_t)、當前步數 (t)、Mamba給的提示 (context)
        # 我們用時間嵌入 (Time Embedding) 讓模型知道現在是哪一步
        self.time_embed = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # 核心 MLP 網路
        self.net = nn.Sequential(
            nn.Linear(pred_length + hidden_dim + context_dim, hidden_dim * 2),
            nn.SiLU(),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, pred_length) # 輸出預測的雜訊
        )

    def forward(self, x_t, t, context):
        """
        x_t: [Batch, Pred_Len] (充滿雜訊的未來股價)
        t: [Batch, 1] (當前的擴散步數，例如 0.5)
        context: [Batch, Context_Dim] (Mamba 提煉的微觀情緒)
        """
        t_emb = self.time_embed(t)
        # 把這三個東西暴力拼接 (Concat) 在一起餵給網路
        nn_input = torch.cat([x_t, t_emb, context], dim=-1)
        return self.net(nn_input)

# ==========================================
# 2. Mamba 提詞器 (Micro Mamba)
# ==========================================
# 負責吃 5 分鐘 K 線，吐出 Context (假設你已經裝了 mamba-ssm)
try:
    from mamba_ssm import Mamba
    HAS_MAMBA = True
except ImportError:
    HAS_MAMBA = False
    print("⚠️ 尚未安裝 mamba-ssm，將使用模擬的 Mamba 進行測試")

class MicroMamba(nn.Module):
    def __init__(self, input_dim=2, d_model=64, context_dim=128):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)

        if HAS_MAMBA:
            self.mamba = Mamba(d_model=d_model, d_state=16, d_conv=4, expand=2)
        else:
            # 如果還沒裝套件，先用 GRU 頂替一下確保程式能跑
            self.mamba = nn.GRU(d_model, d_model, batch_first=True)

        self.output_proj = nn.Linear(d_model, context_dim)

    def forward(self, x):
        x = self.input_proj(x)
        if HAS_MAMBA:
            x = self.mamba(x)
        else:
            x, _ = self.mamba(x)
        # 只取序列最後一步的隱藏狀態作為 Context
        last_step_features = x[:, -1, :]
        return self.output_proj(last_step_features)

# ==========================================
# 3. 終極組裝：MarketMamba Diffusion Wrapper
# ==========================================
class MarketMambaDiffusion(nn.Module):
    def __init__(self, seq_length=270, pred_length=1, input_dim=2, context_dim=128):
        super().__init__()
        self.conditioner = MicroMamba(input_dim, d_model=64, context_dim=context_dim)
        self.denoiser = DenoiseNetwork(pred_length, context_dim, hidden_dim=256)

    def compute_loss(self, x_history, y_future):
        """
        訓練時呼叫這個函數！
        x_history: 過去的 5 分 K 線 [Batch, Seq_Len, 2]
        y_future: 真實的未來走勢 [Batch, Pred_Len]
        """
        batch_size = x_history.shape[0]

        # 1. 讓 Mamba 產生 Context
        context = self.conditioner(x_history)

        # 2. 隨機抽取擴散步數 t (0 ~ 1 之間)
        t = torch.rand(batch_size, 1, device=x_history.device)

        # 3. 產生隨機標準常態雜訊
        noise = torch.randn_like(y_future)

        # 4. 製造 x_t (在真實未來中加入雜訊)
        # 這裡用最簡單的線性排程：x_t = sqrt(1-t)*y_future + sqrt(t)*noise
        x_t = torch.sqrt(1 - t) * y_future + torch.sqrt(t) * noise

        # 5. 讓 Denoise 網路去猜我們加了什麼雜訊
        predicted_noise = self.denoiser(x_t, t, context)

        # 6. 計算 MSE Loss (猜的雜訊 vs 真的雜訊)
        loss = F.mse_loss(predicted_noise, noise)
        return loss

# ==========================================
# 🧪 馬上測試一下形狀對不對！
# ==========================================
if __name__ == "__main__":
    # 假設 Batch Size 是 32
    # x_history 是過去 270 根 5 分 K 線 (2 維特徵)
    dummy_x = torch.randn(32, 270, 2)
    # y_future 是未來 1 步的報酬率
    dummy_y = torch.randn(32, 1)

    model = MarketMambaDiffusion(seq_length=270, pred_length=1)

    # 測試前向傳播與 Loss 計算
    loss = model.compute_loss(dummy_x, dummy_y)
    print(f"✅ 模型組裝成功！計算出的 Loss 值為: {loss.item():.4f}")

RuntimeError: Expected x.is_cuda() to be true, but got false.  (Could this error message be improved?  If so, please report an enhancement request to PyTorch.)

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os

# 1. 讀取真實資料 (假設你已經 mount 了 Google Drive)
# 我們先拿老大哥台積電的 5 分鐘 K 線來開刀
file_path = '/content/drive/MyDrive/MarketMamba_Database/Micro_5min/2330_5m.parquet'

if not os.path.exists(file_path):
    print("❌ 找不到檔案！請檢查 Google Drive 裡有沒有這支股票。")
else:
    print(f"✅ 成功找到台積電真實資料: {file_path}")
    df_real = pd.read_parquet(file_path)

# 2. 宣告專屬的真實資料集 (RealMarketDataset)
class RealMarketDataset(Dataset):
    def __init__(self, df, seq_length=270, pred_length=1):
        # 把我們算好的 Log_Ret (對數報酬) 跟 Log_Vol (對數交易量) 抓出來轉成 Tensor
        features = df[['Log_Ret', 'Log_Vol']].values
        self.features = torch.tensor(features, dtype=torch.float32)
        self.seq_length = seq_length
        self.pred_length = pred_length

    def __len__(self):
        return len(self.features) - self.seq_length - self.pred_length + 1

    def __getitem__(self, idx):
        # 抓取過去 270 根 K 線 (約 1 天)
        x = self.features[idx : idx + self.seq_length]
        # 預測未來 1 根 K 線的「報酬率」(第 0 個 feature 是 Log_Ret)
        y = self.features[idx + self.seq_length : idx + self.seq_length + self.pred_length, 0]
        return x, y

# 3. 啟動 DataLoader
# batch_size=32 代表模型一次看 32 種不同的「歷史情境」
dataset = RealMarketDataset(df_real, seq_length=270)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# 抽一包出來檢查！
x_batch, y_batch = next(iter(dataloader))
print(f"📊 真實資料總筆數: {len(df_real)}")
print(f"📦 X_batch 形狀 (Batch, Seq_Len, Features): {x_batch.shape}")
print(f"📦 Y_batch 形狀 (Batch, Pred_Len): {y_batch.shape}")

✅ 成功找到台積電真實資料: /content/drive/MyDrive/MarketMamba_Database/Micro_5min/2330_5m.parquet
📊 真實資料總筆數: 3132
📦 X_batch 形狀 (Batch, Seq_Len, Features): torch.Size([32, 270, 2])
📦 Y_batch 形狀 (Batch, Pred_Len): torch.Size([32, 1])


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import os
from tqdm.auto import tqdm

class MultiStockMicroDataset(Dataset):
    def __init__(self, data_dir, seq_length=270, pred_length=1):
        """
        data_dir: 你 Google Drive 裡 Micro_5min 的資料夾路徑
        """
        self.seq_length = seq_length
        self.pred_length = pred_length

        # 用來存放所有有效的滑動窗口樣本
        self.x_data = []
        self.y_data = []

        # 取得資料夾下所有的 parquet 檔案清單
        file_list = [f for f in os.listdir(data_dir) if f.endswith('.parquet')]
        print(f"🔍 找到 {len(file_list)} 檔股票資料，開始載入記憶體...")

        # 迴圈讀取每一檔股票
        for file_name in tqdm(file_list, desc="載入全市場資料"):
            file_path = os.path.join(data_dir, file_name)
            try:
                df = pd.read_parquet(file_path)
                features = df[['Log_Ret', 'Log_Vol']].values

                # 如果這檔股票的資料長度連一個完整的窗口都切不出來，就直接丟棄
                total_len = len(features)
                if total_len < (seq_length + pred_length):
                    continue

                # 針對「這單一檔股票」進行滑動窗口切分，確保不會跨股票邊界
                valid_windows = total_len - seq_length - pred_length + 1
                for i in range(valid_windows):
                    # 擷取 X 和 Y，並存入總體 list
                    x_window = features[i : i + seq_length]
                    y_window = features[i + seq_length : i + seq_length + pred_length, 0]

                    self.x_data.append(x_window)
                    self.y_data.append(y_window)
            except Exception as e:
                # 略過損壞的檔案
                pass

        # 最後統一轉換成 PyTorch 的 Tensor，這會大幅提升訓練時讀取的速度
        print(f"⏳ 正在轉換為 Tensor，請稍候...")
        self.x_data = torch.tensor(np.array(self.x_data), dtype=torch.float32)
        self.y_data = torch.tensor(np.array(self.y_data), dtype=torch.float32)
        print(f"✅ 全市場資料載入完成！共切出 {len(self.x_data)} 組訓練樣本。")

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        return self.x_data[idx], self.y_data[idx]

# ==========================================
# 啟動全市場 DataLoader
# ==========================================
# data_dir = '/content/drive/MyDrive/MarketMamba_Database/Micro_5min'
# global_dataset = MultiStockMicroDataset(data_dir, seq_length=270)
# global_dataloader = DataLoader(global_dataset, batch_size=256, shuffle=True)